In [5]:
%cd /kaggle/working
!git clone https://github.com/nhannhan2201/math-instruct-llama.git
%cd /kaggle/working/math-instruct-llama
!git log -1 --oneline
!test -f src/test.py && echo "Đã có src/test.py"

/kaggle/working
fatal: destination path 'math-instruct-llama' already exists and is not an empty directory.
/kaggle/working/math-instruct-llama
333d020 (HEAD -> main, origin/main, origin/HEAD) Add held-out token evaluation for base and adapters
Đã có src/test.py


In [2]:
%pip install -r requirements.txt
%pip uninstall -y torchao

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 110.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 103.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 5.9 MB/s eta 0:00:00
   ━━━━

In [3]:
from pathlib import Path

adapter_metadata = sorted(Path("/kaggle/input").rglob("effective_data.json"))
for metadata in adapter_metadata:
    adapter_dir = metadata.parent
    print("Adapter:", adapter_dir)
    print("  config:", (adapter_dir / "adapter_config.json").is_file())
    print("  weights:", (adapter_dir / "adapter_model.safetensors").is_file())

Adapter: /kaggle/input/datasets/nhannhan22/models/lora_training_5c645da682df/final
  config: True
  weights: True
Adapter: /kaggle/input/datasets/nhannhan22/models/qlora_training_6c775f8d8113/final
  config: True
  weights: True


In [7]:
!CUDA_VISIBLE_DEVICES=0 python -m src.test --method base

model.safetensors: 100%|████████████████████| 2.47G/2.47G [00:05<00:00, 469MB/s]
generation_config.json: 100%|██████████████████| 239/239 [00:00<00:00, 1.06MB/s]
Test: 500 complete samples; longest=1637 tokens; no truncation
Tokenizing eval dataset: 100%|████████| 500/500 [00:01<00:00, 479.03 examples/s]
2026/10/18 05:07:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/18 05:07:02 INFO mlflow.store.db.utils: Updating database tables
2026/10/18 05:07:04 INFO mlflow.tracking.fluent: Experiment with name 'llama-3-math-instruct-test' does not exist. Creating a new experiment.
100%|█████████████████████████████████████████| 500/500 [00:37<00:00, 13.43it/s]
{
  "run_id": "08caf3e2ef064650b5cfcaa69a7b9246",
  "method": "base",
  "test_loss": 1.5333656072616577,
  "test_mean_token_accuracy": 0.6900176354348659
}


In [8]:
!CUDA_VISIBLE_DEVICES=0 python -m src.test \
    --method lora \
    --adapter-path /kaggle/input/datasets/nhannhan22/models/lora_training_5c645da682df/final

Loading weights: 100%|███████████████████████| 146/146 [00:01<00:00, 101.96it/s]
Test: 500 complete samples; longest=1637 tokens; no truncation
100%|█████████████████████████████████████████| 500/500 [00:48<00:00, 10.32it/s]
{
  "run_id": "111d457c10a94b44a3ecdd030beac62a",
  "method": "lora",
  "test_loss": 0.790376603603363,
  "test_mean_token_accuracy": 0.7970716972947121
}


In [9]:
!CUDA_VISIBLE_DEVICES=0 python -m src.test \
    --method qlora \
    --adapter-path /kaggle/input/datasets/nhannhan22/models/qlora_training_6c775f8d8113/final

Loading weights:   1%|▏                         | 1/146 [00:00<01:08,  2.11it/s]/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|████████████████████████| 146/146 [00:01<00:00, 85.11it/s]
Test: 500 complete samples; longest=1637 tokens; no truncation
100%|█████████████████████████████████████████| 500/500 [01:04<00:00,  7.80it/s]
{
  "run_id": "d4c434a7fb3443d8aa25ff883cb15781",
  "method": "qlora",
  "test_loss": 0.811022937297821,
  "test_mean_token_accuracy": 0.7940802190899849
}


In [10]:
import shutil

shutil.make_archive(
    "/kaggle/working/mlruns-test",
    "zip",
    root_dir="/kaggle/working/math-instruct-llama",
    base_dir="mlruns",
)

'/kaggle/working/mlruns-test.zip'